**STEP 0: Library Perequisites**

---



**STEP 1.1: Loading model directly using Ollama Library**

---



In [1]:
import ollama

print("Asking Ollama (with confidence scores)...")
response = ollama.chat(
    model='llama3:instruct',
    messages=[
        {'role': 'user', 'content': 'Who are you?'}
    ],
    options={
        'temperature': 0  
    },
    logprobs=True     
)

print(f"Answer: {response['message']['content']}\n")

if 'logprobs' in response:
    print("--- Confidence Scores (First 5 Tokens) ---")
    for token_data in response['logprobs'][:5]:
        token_str = token_data.get('token', '')
        score = token_data.get('logprob', -999) 
        
        # A score close to 0 (e.g., -0.01) is VERY confident.
        # A score like -2.5 is LOW confidence.
        print(f"Token: '{token_str}' | Logprob: {score:.4f}")
else:
    print("Logprobs not found. Ensure your Ollama server version is up to date.")

Asking Ollama (with confidence scores)...
Answer: I am LLaMA, an AI assistant developed by Meta AI that can understand and respond to human input in a conversational manner. I'm not a human, but a computer program designed to simulate conversation and answer questions to the best of my ability based on my training data.

My primary function is to provide information and assist with tasks through text-based conversations. I can generate human-like responses to a wide range of topics, including but not limited to:

1. General knowledge: History, science, technology, culture, and more.
2. Language translation: I can translate text from one language to another.
3. Writing assistance: I can help with writing tasks such as proofreading, suggesting alternative phrases, or even generating text based on a prompt.
4. Conversation: I can engage in natural-sounding conversations, using context and understanding to respond to questions and statements.

I'm constantly learning and improving my abili

**STEP 1.2: Loading model directly using OpenAI library Wrapper**

---



In [ ]:
from openai import OpenAI

# 1. Initialize the Client to hijack the local Ollama server
client = OpenAI(
    base_url='http://localhost:11434/v1',  # Points to your local machine
    api_key='ollama'                       # Required string, but ignored by Ollama
)

print("Asking Ollama (via OpenAI Wrapper for detailed logprobs)...")

try:
    response = client.chat.completions.create(
        model="llama3:instruct",
        messages=[
            {'role': 'user', 'content': 'Who are you?'}
        ],
        temperature=0,      # 0 = Deterministic (Scientific standard)
        logprobs=True,      # Turn on confidence scores
        top_logprobs=5      # <--- THE KEY FEATURE: Get the top 5 alternatives per token
    )

    answer_text = response.choices[0].message.content
    print(f"Answer: {answer_text}\n")

    # We inspect the first n tokens to see if the model hesitated.
    logprob_data = response.choices[0].logprobs.content
    
    if logprob_data:
        print("--- Problogs: Confidence & Alternatives ---")
        PROBLOGS_COUNT = 5 # How many tokens to inspect
        for i, token_data in enumerate(logprob_data[:PROBLOGS_COUNT]):
            token_str = token_data.token
            score = token_data.logprob
            
            print(f"\nToken {i+1}: '{token_str}' | Score: {score:.4f}")
            
            if token_data.top_logprobs:
                print("   Alternatives considered:")
                for alt in token_data.top_logprobs:
                    print(f"   - '{alt.token}': {alt.logprob:.4f}")
    else:
        print("No logprobs returned. Check if your Ollama version supports the v1 API.")

except Exception as e:
    print(f"Error: {e}")
    print("Ensure 'pip install openai' is run and Ollama is running.")

Asking Ollama (via OpenAI Wrapper for detailed logprobs)...
Error: Error code: 404 - {'error': {'message': "model 'llama3:instruct4' not found", 'type': 'api_error', 'param': None, 'code': None}}
Ensure 'pip install openai' is run and Ollama is running.


**STEP 2: Define a string variable that contains a Prolog database**

___

In [ ]:
import janus_swi